# XAI estructure
Edición de las rutas y confirmación del cache correspondiente al checkpoint y a su preprocesamiento.
Las funciones generales están en `xai_core.py`; los imports de aplicación deben estar con el formato `model_adapter.py`.


In [ ]:
from pathlib import Path
import json
import numpy as np
import torch
from xai_core import (predict, binary_groups, saliency, integrated_gradients,
                      occlusion, kernel_shap, save_results,
                      plot_importance, plot_signal_map)
from sanchez_adapter import load_sanchez, load_cached

# Rutas dentro de WSL. REPO_ROOT debe contener pt/pipeline.py.
REPO_ROOT = Path('/CAMBIAR/sanchez_repository')
CONFIG_PATH = REPO_ROOT / 'api/networks/pt/config_used.json'
MODEL_PATH = REPO_ROOT / 'api/networks/pt/model_final.pt'
CACHE_DIR = REPO_ROOT / 'results'
OUTPUT_DIR = Path('results_xai_general')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
THRESHOLD = 0.5
N_PER_GROUP = 3
# Use valores definidos según el espacio de entrada del modelo.
SIGNAL_BASELINE = 0.0
FEATURE_BASELINE = 0.0
RUN_SHAP = False  # Active cuando indique un background adecuado.
BACKGROUND_PATH = None  # .npy (N,F), features ya escaladas del conjunto de referencia
CACHE_PREPROCESS_CONFIRMED = False  # cambie a True tras comprobar procedencia


ModuleNotFoundError: No module named 'notebooks'

In [ ]:
if not CACHE_PREPROCESS_CONFIRMED:
    raise ValueError('Confirme que el cache usa el preprocesamiento, orden de canales y scaler del modelo.')
x, features, labels, old_metadata = load_cached(CACHE_DIR)
adapter, config = load_sanchez(REPO_ROOT, CONFIG_PATH, MODEL_PATH,
                               x.shape[1:], features.shape[1], DEVICE)
feature_names = config['selected_features']
# Cambie por los nombres EN EL ORDEN REAL del loader. No inferimos el montaje.
channel_names = [f'Canal {i+1}' for i in range(x.shape[2])]
probabilities = predict(adapter.predict, (x, features), batch_size=32)
groups = binary_groups(labels, probabilities, THRESHOLD, N_PER_GROUP)
print({name: idx.tolist() for name, idx in groups.items()})
# Registros con crisis + ventanas extremas: análisis de casos, no evaluación global.


In [ ]:
arrays = {'probabilities': probabilities, 'labels': labels}
if RUN_SHAP:
    if BACKGROUND_PATH is None:
        raise ValueError('Indique BACKGROUND_PATH; no se usan automáticamente los casos a explicar.')
    background = np.load(BACKGROUND_PATH, allow_pickle=False)

for group, indices in groups.items():
    if len(indices) == 0:
        continue
    inputs = (np.asarray(x[indices]), np.asarray(features[indices]))
    arrays[f'{group}_indices'] = indices
    for input_index, name, baseline in [(0, 'signal', SIGNAL_BASELINE),
                                         (1, 'features', FEATURE_BASELINE)]:
        prefix = f'{group}_{name}'
        arrays[prefix + '_saliency'] = saliency(adapter, inputs, input_index)
        attrs, delta = integrated_gradients(
            adapter, inputs, input_index, baseline=baseline,
            n_steps=50, internal_batch_size=len(indices))
        arrays[prefix + '_ig'] = attrs
        arrays[prefix + '_ig_delta'] = delta
        arrays[prefix + '_occlusion'], _ = occlusion(
            adapter.predict, inputs, input_index, baseline=baseline)
        print(group, name, 'máximo |delta IG|:', float(np.abs(delta).max()))
    if RUN_SHAP:
        phi, expected, residual = kernel_shap(
            adapter.predict, inputs, 1, background, nsamples=100)
        arrays[f'{group}_features_shap'] = phi
        arrays[f'{group}_shap_expected'] = expected
        arrays[f'{group}_shap_residual'] = residual
    # Guarda después de cada grupo completado.
    save_results(OUTPUT_DIR, arrays, {
        'model_path': str(MODEL_PATH), 'config_path': str(CONFIG_PATH),
        'cache_dir': str(CACHE_DIR), 'threshold': THRESHOLD,
        'n_per_group': N_PER_GROUP, 'fs': config['target_fs'],
        'channel_names': channel_names, 'feature_names': feature_names,
        'score': 'probabilidad binaria; selector de Sánchez',
        'signal_baseline': SIGNAL_BASELINE, 'feature_baseline': FEATURE_BASELINE,
        'shap_background': str(BACKGROUND_PATH), 'config': config,
        'note': 'Atribuciones parciales con otras entradas fijas; no causales.'})


## Regenerar PDF sin repetir XAI
En otra sesión basta con ejecutar la siguiente celda (y tener `xai_core.py` al lado).
No necesita cargar el modelo ni ejecutar las celdas anteriores.


In [ ]:
from pathlib import Path
import json
import numpy as np
from xai_core import plot_importance, plot_signal_map

output = Path('results_xai_general')  # ajustar si cambió OUTPUT_DIR
meta = json.loads((output / 'metadata.json').read_text(encoding='utf-8'))
with np.load(output / 'attributions.npz', allow_pickle=False) as saved:
    for group in ['TP', 'FP', 'TN', 'FN']:
        if f'{group}_indices' not in saved:
            continue
        for method in ['saliency', 'ig', 'occlusion', 'shap']:
            key = f'{group}_features_{method}'
            if key in saved:
                plot_importance(saved[key], meta['feature_names'],
                                output / f'{key}.pdf', title=f'{group}: {method}',
                                absolute=(method != 'occlusion'))
        plot_importance(saved[f'{group}_signal_occlusion'], meta['channel_names'],
                        output / f'{group}_channels.pdf',
                        title=f'{group}: oclusión de canales', absolute=False)
        for i, original in enumerate(saved[f'{group}_indices']):
            plot_signal_map(saved[f'{group}_signal_ig'][i], meta['channel_names'],
                            meta['fs'], output / f'{group}_window_{original}_ig.pdf',
                            title=f'{group}: ventana {original}, IG parcial')


## Otro modelo
Para oclusión/Kernel SHAP, proporcione una función `score(*arrays)` que devuelva `(B,)`.
Ejemplo scikit-learn: `score = lambda x: classifier.predict_proba(x)[:, 1]`;
`occlusion(score, (X,), input_index=0, baseline=referencia)`.
Para saliency/IG use `TorchAdapter(model, selector, device)` y una tupla de entradas.
El selector elige explícitamente una clase/probabilidad/logit. TensorFlow necesita
su propio backend de gradientes; puede usar el núcleo de predicción sin PyTorch.
